In [ ]:
!pip install torch torchvision

Ucitajmo sve potrebne biblioteke

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

import scipy.io as sio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ucitavamo pretreniran ResNet50 model (ImageNet tezine)
model = models.resnet50(weights="IMAGENET1K_V1")
model.eval()

# DOBIJANJE SPEKTOGRAMA IZ .MAT SIGNALA
(dio koji radim sama from scratch)

In [ ]:
def stft(signal, vel_prozora, pomak):
    # broj okvira
    M = 1 + (len(signal) - vel_prozora) // pomak
    N = vel_prozora

    # matrica rezultata
    X = np.empty((N // 2 + 1, M), dtype=complex)

    # n se krece od 0 do N-1
    n = np.arange(N)

    for m in range(M):
        # izaberemo okvir i pomnozimo sa prozorom
        okvir = signal[m * pomak: m * pomak + N] * np.hamming(N)

        # suma za svako k
        for k in range(N // 2 + 1):
            # e^{-j2*pi*k*n/N}
            eksp = np.exp(-1j * 2 * np.pi * k * n / N)
            # suma x(n) * w(n) * e^{-j2*pi*k*n/N}
            X[k, m] = np.sum(okvir * eksp)

    return X

In [ ]:
# ucitavanje primjera signala iz .mat fajla
# (https://www.alireza-zarrinmehr.com/how-to-read-and-write-matlab-files-in-python/)
putanja_signali = "C:/Users/Korisnik/Desktop/UDOS/midterm3/training2017"
signal = sio.loadmat(f"{putanja_signali}/A00001.mat")["val"][0]

# parametri STFT-a
vel_prozora = 256
pomak = 128

# provjera moje implementacije poredjenjem sa scipy STFT-om
import scipy.signal as ss

X_moja = stft(signal, vel_prozora, pomak)
f, t, X_scipy = ss.stft(signal, fs=300, window="hamming",
                         nperseg=vel_prozora, noverlap=vel_prozora - pomak)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.imshow(np.abs(X_moja), aspect="auto", origin="lower")
ax1.set_title("Moja STFT")
ax1.set_xlabel("Okviri")
ax1.set_ylabel("Frekvencija (bin)")

ax2.imshow(np.abs(X_scipy), aspect="auto", origin="lower")
ax2.set_title("Scipy STFT")
ax2.set_xlabel("Okviri")
ax2.set_ylabel("Frekvencija (bin)")

plt.tight_layout()
plt.show()

### Ucitavanje labela i podjela na trening/validacijski skup

In [ ]:
from sklearn.model_selection import train_test_split

# fajl sa labelama (ime signala, klasa: A = AFib, N = Normalan, O = Other, ~ = Zasumljen)
labele = pd.read_csv(f"{putanja_signali}/REFERENCE.csv", header=None, names=["ime", "klasa"])

# podjela na trening (80%) i validacijski (20%) skup, stratifikovano po klasi
trening, validacija = train_test_split(
    labele, test_size=0.2, stratify=labele["klasa"], random_state=42
)

print("Trening:", len(trening), "| Validacija:", len(validacija))

In [ ]:
# Pravljenje i spremanje spektrogram
from PIL import Image
import os

putanja_dataset = "C:/Users/Korisnik/Desktop/UDOS/midterm3/dataset"

# pravimo foldere za svaku klasu (trening i validacija)
for folder in ["trening", "validacija"]:
    for klasa in labele["klasa"].unique():
        os.makedirs(f"{putanja_dataset}/{folder}/{klasa}", exist_ok=True)

ukupno = len(labele)

for index, red in labele.iterrows():
    ime = red["ime"]
    klasa = red["klasa"]

    # odredjujemo da li signal ide u trening ili validacijski folder
    folder = "trening" if ime in trening["ime"].values else "validacija"

    putanja_mat = f"{putanja_signali}/{ime}.mat"
    putanja_slika = f"{putanja_dataset}/{folder}/{klasa}/{ime}.png"

    # ucitaj signal i izracunaj spektrogram
    signal = sio.loadmat(putanja_mat)["val"][0]
    X = stft(signal, vel_prozora, pomak)

    # pretvori u dB skalu
    spektrogram_db = 20 * np.log10(np.abs(X) + 1e-10)
    spektrogram_db = np.clip(spektrogram_db, a_min=-80, a_max=None)

    # spremi kao 224x224 RGB sliku
    fig = plt.figure(figsize=(2.24, 2.24), dpi=100, frameon=False)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(spektrogram_db, aspect="auto", origin="lower", cmap="inferno")
    ax.axis("off")
    plt.savefig(putanja_slika, dpi=100, bbox_inches=None, pad_inches=0)
    plt.close()

print("Gotovo!")

## FINE - TUNING ResNet50

In [ ]:
# transformacija slika (resize + tenzor) za ulaz u ResNet50
transformacije = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ucitavanje dataseta iz foldera (ImageFolder klase uzima iz naziva podfoldera)
trening_dataset = ImageFolder(f"{putanja_dataset}/trening", transform=transformacije)
validacija_dataset = ImageFolder(f"{putanja_dataset}/validacija", transform=transformacije)

trening_loader = DataLoader(trening_dataset, batch_size=32, shuffle=True)
validacija_loader = DataLoader(validacija_dataset, batch_size=32, shuffle=False)

# zamjena izlaznog sloja da odgovara broju nasih klasa
broj_klasa = len(trening_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, broj_klasa)

print("Klase:", trening_dataset.classes)

In [ ]:
# Trening modela (treniramo samo izlazni sloj)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
kriterij = nn.CrossEntropyLoss()

broj_epoha = 8

for epoha in range(broj_epoha):
    model.train()
    ukupan_loss = 0

    for slike, oznake in trening_loader:
        optimizer.zero_grad()
        izlaz = model(slike)
        loss = kriterij(izlaz, oznake)
        loss.backward()
        optimizer.step()
        ukupan_loss += loss.item()

    # validacija nakon svake epohe
    model.eval()
    tacno = 0
    ukupno = 0

    with torch.no_grad():
        for slike, oznake in validacija_loader:
            izlaz = model(slike)
            _, predikcije = torch.max(izlaz, 1)
            tacno += (predikcije == oznake).sum().item()
            ukupno += oznake.size(0)

    tacnost = 100 * tacno / ukupno
    print(f"Epoha {epoha+1}/{broj_epoha} — loss: {ukupan_loss:.4f} — tacnost: {tacnost:.2f}%")

# spremanje modela
torch.save(model.state_dict(), "C:/Users/Korisnik/Desktop/UDOS/midterm3/resnet50_ekg_ep8.pth")
print("Gotovo!")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# uredjaj na kojem radimo (GPU ako je dostupan, inace CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# nazivi klasa (mora biti isti redoslijed kao tokom treninga)
try:
    CLASS_NAMES = list(trening_dataset.classes)
except NameError:
    CLASS_NAMES = ["A", "N", "O", "~"]

num_classes = len(CLASS_NAMES)

# arhitektura mora biti ista kao tokom treninga
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# ucitavanje snimljenih tezina
MODEL_PATH = "C:/Users/Korisnik/Desktop/UDOS/midterm3/resnet50_ekg_ep8.pth"
checkpoint = torch.load(MODEL_PATH, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict):
    model.load_state_dict(checkpoint)
else:
    model = checkpoint

model = model.to(device)
model.eval()

print("Gotovo!")